In [0]:
from TornAPI.Torn import Faction
from pyspark.sql.functions import lit, explode, from_unixtime, get
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, ArrayType

import datetime as dt

faction_api = Faction(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:
if spark.catalog.tableExists("torn.faction.crimes"):
    df = spark.read.table("torn.faction.crimes")
    drop_id_list = [id[0] for id in df.select("id").where("status in ('Recruiting', 'Planning')").collect()]
    df = df.where("status NOT in ('Recruiting', 'Planning')")
    df.write.format("delta").mode("overwrite").saveAsTable("torn.faction.crimes")
    last_date = df.select("created_at").agg({"created_at":"max"}).collect()[0]["max(created_at)"]
    id_list = [id[0] for id in df.select("id").collect()]
else:
    last_date = 1681484237
    id_list = []



In [0]:
data = faction_api.get_crimes(ts_from=last_date)

schema = StructType([
    StructField("id", IntegerType()),
    StructField("previous_crime_id", IntegerType()),
    StructField("name", StringType()),
    StructField("difficulty", IntegerType()),
    StructField("status", StringType()),
    StructField("created_at", IntegerType()),
    StructField("planning_at", IntegerType()),
    StructField("executed_at", IntegerType()),
    StructField("ready_at", IntegerType()),
    StructField("expired_at", IntegerType()),
    StructField("slots", ArrayType(
        StructType([
            StructField("position", StringType()),
                    StructField("position_id", StringType()),
        StructField("position_number", IntegerType()),
        StructField("item_requirement", StructType([
            StructField("id", IntegerType()),
            StructField("is_reusable", BooleanType()),
            StructField("is_available", BooleanType())
            ])),
        StructField("user", StructType([
            StructField("outcome", StringType()),
            StructField("id", IntegerType()),
            StructField("joined_at", IntegerType()),
            StructField("progress", IntegerType()),
            StructField("item_outcome", StringType())
        ])),
        StructField("checkpoint_pass_rate", IntegerType())
        ])
    )),
    StructField("rewards", StructType([
        StructField("money", IntegerType()),
        StructField("items", ArrayType(
            StructType([
                StructField("id", IntegerType()),
                StructField("quantity", IntegerType())
            ]))),
            StructField("respect", IntegerType()),
            StructField("scope", IntegerType()),
            StructField("payout", StructType([
                StructField("type", StringType()),
                StructField("percentage", IntegerType()),
                StructField("paid_by", IntegerType()),
                StructField("paid_at", IntegerType())
            ]))
        ]))
    ])


sp_crimes = spark.createDataFrame(data["crimes"], schema= schema)

sp_crimes = sp_crimes.filter(~sp_crimes.id.isin(id_list))

sp_crimes.write.format("delta").mode("append").saveAsTable("torn.faction.crimes")